# Global SUVR branch check (single-visit labels)

This notebook follows the plotting style in `analysis/figures4paper.ipynb`, but focuses on the single-visit file `trial2_single_pre.csv`.

Note: the CSV does not contain an explicit global SUVR column. Here I use the row-wise mean of all cortical `CTX_*` SUVR features as a proxy global SUVR.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D

plt.style.use('default')
sns.set_palette('muted')

FIGSIZE = (3, 3)
DPI = 200
TITLE_SIZE = 11

LABELS = {
    1: 'CN',
    2: 'MCI',
    4: 'AD',
}

BRANCH_LINE_COLORS = {
    'common_branch': 'grey',
    'AD_branch': 'red',
    'normal_branch': 'blue',
}

BRANCH_POINT_COLORS = {
    'common_branch': '#8c8c8c',
    'AD_branch': '#d62728',
    'normal_branch': '#1f77b4',
}

DIAG_ORDER = [1, 2, 4]
NEAREST_DIFF_QUANTILE = 0.05
MAX_PLOT_PAIRS = 20


In [ ]:
df = pd.read_csv('./trial2_train_pre.csv')
df = df[df['lb'].isin(DIAG_ORDER)].copy().reset_index(drop=True)

suvr_cols = [col for col in df.columns if col.startswith('CTX_')]
df['global_suvr'] = df[suvr_cols].mean(axis=1)
df['row_id'] = np.arange(len(df))

print(f'Rows used: {len(df)}')
print(f'Unique RIDs: {df["RID"].nunique()}')
print(f'Cortical SUVR columns used for global SUVR proxy: {len(suvr_cols)}')
print()
print('Diagnosis counts:')
print(df['lb'].value_counts().sort_index())
print()
print('Branch counts:')
print(df['branch'].value_counts())
print()
print('Global SUVR range:')
print(df['global_suvr'].agg(['min', 'max', 'mean', 'std']))


In [ ]:
def build_cluster_centers(dataframe):
    return dataframe.groupby(['cluster', 'branch'], as_index=False)[['embedding1', 'embedding2']].mean()


def plot_trajectory_background(ax, dataframe, cluster_centers):
    ax.scatter(
        dataframe['embedding1'],
        dataframe['embedding2'],
        c='lightgrey',
        alpha=0.25,
        s=8,
        linewidths=0,
    )

    for branch, branch_df in cluster_centers.groupby('branch'):
        branch_df = branch_df.sort_values('cluster')
        ax.plot(
            branch_df['embedding1'],
            branch_df['embedding2'],
            color=BRANCH_LINE_COLORS[branch],
            linewidth=1.2,
            alpha=0.9,
        )

    ax.set_xticks([])
    ax.set_yticks([])


def get_nearest_cross_branch_diffs(dataframe):
    rows = []

    for row in dataframe.itertuples(index=False):
        others = dataframe[dataframe['branch'] != row.branch].copy()
        others['diff'] = (others['global_suvr'] - row.global_suvr).abs()
        nearest = others.nsmallest(1, 'diff').iloc[0]

        rows.append({
            'row_id': row.row_id,
            'RID': int(row.RID),
            'lb': int(row.lb),
            'branch': row.branch,
            'global_suvr': float(row.global_suvr),
            'nearest_row_id': int(nearest['row_id']),
            'nearest_RID': int(nearest['RID']),
            'nearest_lb': int(nearest['lb']),
            'nearest_branch': nearest['branch'],
            'nearest_global_suvr': float(nearest['global_suvr']),
            'diff': float(nearest['diff']),
        })

    return pd.DataFrame(rows).sort_values('diff').reset_index(drop=True)


def build_cross_branch_pairs(dataframe, tolerance):
    pair_rows = []

    for left_id in range(len(dataframe)):
        left = dataframe.iloc[left_id]

        for right_id in range(left_id + 1, len(dataframe)):
            right = dataframe.iloc[right_id]
            if left['branch'] == right['branch']:
                continue

            diff = abs(left['global_suvr'] - right['global_suvr'])
            if diff <= tolerance:
                pair_rows.append({
                    'left_row_id': int(left['row_id']),
                    'right_row_id': int(right['row_id']),
                    'left_RID': int(left['RID']),
                    'right_RID': int(right['RID']),
                    'left_lb': int(left['lb']),
                    'right_lb': int(right['lb']),
                    'left_branch': left['branch'],
                    'right_branch': right['branch'],
                    'left_global_suvr': float(left['global_suvr']),
                    'right_global_suvr': float(right['global_suvr']),
                    'abs_diff': float(diff),
                })

    pair_df = pd.DataFrame(pair_rows).sort_values('abs_diff').reset_index(drop=True)
    if not pair_df.empty:
        pair_df['pair_id'] = np.arange(1, len(pair_df) + 1)

    return pair_df


def points_from_pairs(dataframe, pair_df):
    if pair_df.empty:
        return dataframe.iloc[0:0].copy()

    point_ids = np.unique(pair_df[['left_row_id', 'right_row_id']].to_numpy().ravel())
    return dataframe[dataframe['row_id'].isin(point_ids)].copy()


def pretty_branch(branch):
    if branch == 'common_branch':
        return 'common / split point'
    return branch.replace('_branch', '')


def plot_pairs_on_trajectory(dataframe, pair_df, title, max_pairs=MAX_PLOT_PAIRS):
    cluster_centers = build_cluster_centers(dataframe)
    plot_pairs = pair_df.head(max_pairs).copy()
    highlight_df = points_from_pairs(dataframe, plot_pairs)

    fig, ax = plt.subplots(figsize=(FIGSIZE[0] * 1.6, FIGSIZE[1] * 1.1), dpi=DPI)
    plot_trajectory_background(ax, dataframe, cluster_centers)

    for row in plot_pairs.itertuples(index=False):
        left = dataframe.loc[dataframe['row_id'] == row.left_row_id].iloc[0]
        right = dataframe.loc[dataframe['row_id'] == row.right_row_id].iloc[0]

        ax.plot(
            [left['embedding1'], right['embedding1']],
            [left['embedding2'], right['embedding2']],
            linestyle='--',
            linewidth=0.8,
            color='black',
            alpha=0.35,
        )

    for branch, branch_points in highlight_df.groupby('branch'):
        ax.scatter(
            branch_points['embedding1'],
            branch_points['embedding2'],
            c=BRANCH_POINT_COLORS[branch],
            s=10,
            edgecolor='black',
            linewidth=0.5,
            label=pretty_branch(branch),
        )

    ax.set_title(title, fontsize=TITLE_SIZE)
    handles, labels = ax.get_legend_handles_labels()
    dedup = dict(zip(labels, handles))
    ax.legend(dedup.values(), dedup.keys(), fontsize=7, loc='upper right')
    fig.tight_layout()
    plt.show()


def plot_diagnosis_groups(dataframe, pair_df, max_pairs=MAX_PLOT_PAIRS):
    cluster_centers = build_cluster_centers(dataframe)
    plot_pairs = pair_df.head(max_pairs).copy()
    highlight_df = points_from_pairs(dataframe, plot_pairs)

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(FIGSIZE[0] * 3.4, FIGSIZE[1] * 1.05),
        dpi=DPI,
        sharex=True,
        sharey=True,
    )

    for ax, lb in zip(axes, DIAG_ORDER):
        plot_trajectory_background(ax, dataframe, cluster_centers)
        subset = highlight_df[highlight_df['lb'] == lb]

        for branch, branch_points in subset.groupby('branch'):
            ax.scatter(
                branch_points['embedding1'],
                branch_points['embedding2'],
                c=BRANCH_POINT_COLORS[branch],
                s=70,
                edgecolor='black',
                linewidth=0.5,
                label=pretty_branch(branch),
            )

        ax.set_title(f'{LABELS[lb]} (n={len(subset)})', fontsize=TITLE_SIZE)

    handles = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor=BRANCH_POINT_COLORS[branch], markeredgecolor='black', markersize=6, label=pretty_branch(branch))
        for branch in ['common_branch', 'normal_branch', 'AD_branch']
    ]
    axes[-1].legend(handles=handles, fontsize=7, loc='upper right')
    fig.tight_layout()
    plt.show()


In [ ]:
nearest_df = get_nearest_cross_branch_diffs(df)
tolerance = nearest_df['diff'].quantile(NEAREST_DIFF_QUANTILE)
pair_df = build_cross_branch_pairs(df, tolerance)
highlight_df = points_from_pairs(df, pair_df)

print(f'Nearest cross-branch diff quantile used: q={NEAREST_DIFF_QUANTILE:.2f}')
print(f'Tolerance on proxy global SUVR: {tolerance:.6f}')
print(f'Pairs within tolerance: {len(pair_df)}')
print(f'Unique highlighted points: {len(highlight_df)}')

pair_summary = pair_df[[
    'pair_id',
    'left_RID',
    'left_lb',
    'left_branch',
    'left_global_suvr',
    'right_RID',
    'right_lb',
    'right_branch',
    'right_global_suvr',
    'abs_diff',
]].copy()

pair_summary['left_diag'] = pair_summary['left_lb'].map(LABELS)
pair_summary['right_diag'] = pair_summary['right_lb'].map(LABELS)

pair_summary = pair_summary[[
    'pair_id',
    'left_RID',
    'left_diag',
    'left_branch',
    'left_global_suvr',
    'right_RID',
    'right_diag',
    'right_branch',
    'right_global_suvr',
    'abs_diff',
]]

pair_summary.head(MAX_PLOT_PAIRS)


In [ ]:
plot_pairs_on_trajectory(
    df,
    pair_df,
    title='Single-visit points with similar global SUVR but different branches',
    max_pairs=MAX_PLOT_PAIRS,
)

plot_diagnosis_groups(df, pair_df, max_pairs=MAX_PLOT_PAIRS)
